# Route Planner notebook - Group AB



### Walkthrough:


[Go to Environment setup](#Environment) - Requirements, imports, and connection to the cluster

[Go to Data loading and preprocessing](#data-loading-and-preprocessing) - loading dataframes and preprocessing them to be used later.

[Go to Data Delay Calculation](#delay-calculation) - creating a model to predict delays on a route with actual data.

[Go to Graph Building](#building-the-graph) - building a graph to be used by dijkstra

[Go to Dijkstra route finding](#dijkstra) - The algorithm that plans the route based on delays and user input.

[Go to Post-Processing](#post-processing) - Processing the User input and Dijsktra output in order for the UI to function

[Go to UI and Applet](#ui-and-applet) - Code for UI as well as runnable applet. 

## Environment

Please install the necessary packages by running the cell below. 

In [ ]:
# Install required packages from requirements.txt
!pip install -r requirements.txt

### Imports
If any of these imports fail, please double check that you have installed every package from requirements.txt

In [20]:
import os
import re
import ast

import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

import pandas as pd
from pandas import DataFrame
from datetime import timedelta
from typing import Dict, List, Tuple, Any

import plotly.express as px
import heapq
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from pyhive import hive
from functools import reduce
from pyspark.sql.functions import lit
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import unix_timestamp
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import hour, dayofweek, when, col, count
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import LogisticRegression
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType
from pyspark.ml.clustering import PowerIterationClustering
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql import functions as F

### Connection to the cluster

In [ ]:

default_db = 'com490'
hive_server = os.environ.get('HIVE_SERVER','iccluster080.iccluster.epfl.ch:10000')
hadoop_fs = os.environ.get('HADOOP_DEFAULT_FS','hdfs://iccluster067.iccluster.epfl.ch:8020')
username  = os.environ.get('USER', 'anonym')
(hive_host, hive_port) = hive_server.split(':')

conn = hive.connect(
    host=hive_host,
    port=hive_port,
    username=username
)

# create cursor
cur = conn.cursor()

print(f"hadoop hdfs URL is {hadoop_fs}")
print(f"your username is {username}")
print(f"you are connected to {hive_host}:{hive_port}")

## Data loading and Preprocessing

We can choose a region of interest by changing the Object_id value below to whichever desired region. 

In [ ]:
OBJECT_ID = 1 # Choose the region of interest


"""
1 : Lausanne Region
1696 : Fribourg region
...

"""

In [ ]:
# Enable ESRI UDF
cur.execute(f"""
ADD JARS
    {hadoop_fs}/data/jars/esri-geometry-api-2.2.4.jar
    {hadoop_fs}/data/jars/spatial-sdk-hive-2.2.0.jar
    {hadoop_fs}/data/jars/spatial-sdk-json-2.2.0.jar
""")
cur.execute("CREATE TEMPORARY FUNCTION ST_Point AS 'com.esri.hadoop.hive.ST_Point'")
cur.execute("CREATE TEMPORARY FUNCTION ST_Distance AS 'com.esri.hadoop.hive.ST_Distance'")
cur.execute("CREATE TEMPORARY FUNCTION ST_SetSRID AS 'com.esri.hadoop.hive.ST_SetSRID'")
cur.execute("CREATE TEMPORARY FUNCTION ST_GeodesicLengthWGS84 AS 'com.esri.hadoop.hive.ST_GeodesicLengthWGS84'")
cur.execute("CREATE TEMPORARY FUNCTION ST_LineString AS 'com.esri.hadoop.hive.ST_LineString'")
cur.execute("CREATE TEMPORARY FUNCTION ST_AsBinary AS 'com.esri.hadoop.hive.ST_AsBinary'")
cur.execute("CREATE TEMPORARY FUNCTION ST_PointFromWKB AS 'com.esri.hadoop.hive.ST_PointFromWKB'")
cur.execute("CREATE TEMPORARY FUNCTION ST_GeomFromWKB AS 'com.esri.hadoop.hive.ST_GeomFromWKB'")
cur.execute("CREATE TEMPORARY FUNCTION ST_Contains AS 'com.esri.hadoop.hive.ST_Contains'")

In [ ]:
# Create the tables in the format we want
username = "groupab"
#cur.execute(f"""DROP TABLE {username}.sbb_stops_final_region""")
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {username}.sbb_stops_final_region
(
    stop_id STRING,
    stop_name STRING,
    stop_lat FLOAT,
    stop_lon FLOAT
)
STORED AS PARQUET
""")

#cur.execute(f"""DROP TABLE {username}.sbb_stop_times_final_region""")
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {username}.sbb_stop_times_final_region
(
    trip_id STRING,
    stop_id STRING,
    arrival_time STRING,
    departure_time STRING
)
STORED AS PARQUET
""")

#cur.execute(f"""DROP TABLE {username}.sbb_stop_to_stop_final_region""")
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {username}.sbb_stop_to_stop_final_region
(
    stop_id_a STRING,
    stop_id_b STRING,
    distance FLOAT
)
STORED AS PARQUET
""")

In [ ]:
#cur.execute(f"""
#SELECT *
#FROM
#    groupab.sbb_stops_final_region limit 5
#""")
#cur.fetchall()

In [ ]:
# Select all the stops in the region of interest (for example Lausanne region if OBJECT_ID =1)
cur.execute(f"""
INSERT OVERWRITE TABLE {username}.sbb_stops_final_region
SELECT stop_id, stop_name, stop_lat, stop_lon
FROM
    {default_db}.sbb_orc_stops stops JOIN {default_db}.geo_shapes shapes
    WHERE ST_Contains(shapes.geometry, ST_Point(stops.stop_lon,stops.stop_lat)) 
    AND objectid={OBJECT_ID}
""")

In [ ]:
# Select all the pairs of stop within walking (500 meters) distance of each other
cur.execute(f"""
INSERT OVERWRITE TABLE {username}.sbb_stop_to_stop_final_region
SELECT stop_id_a, stop_id_b, distance FROM
(
SELECT
    a.stop_id AS stop_id_a, b.stop_id AS stop_id_b,
    ST_GeodesicLengthWGS84(
        ST_SetSRID(ST_LineString(a.stop_lon, a.stop_lat, b.stop_lon, b.stop_lat), 4326)
    ) AS distance
FROM {username}.sbb_stops_final_region a JOIN {username}.sbb_stops_final_region b
WHERE
    a.stop_id != b.stop_id
) tmp WHERE distance < 500
""")

In [ ]:
# Select the stop times
query=f"""
INSERT OVERWRITE TABLE {username}.sbb_stop_times_final_region
SELECT
    trip_id,
    stop_id,
    departure_time,
    arrival_time
FROM {default_db}.sbb_orc_stop_times
WHERE stop_id IN (SELECT DISTINCT stop_id FROM {username}.sbb_stops_final_region)
AND   trip_id IN (SELECT DISTINCT b.trip_id
        FROM {default_db}.sbb_orc_calendar a INNER JOIN {default_db}.sbb_orc_trips b
        WHERE a.monday='TRUE' AND b.service_id = a.service_id)
"""
cur.execute(query)

In [ ]:
# To store in a local pd dataframe 


def rename_col(df):
    new_names = [col.split('.')[1] for col in df.columns ]
    df.columns = new_names

stops_df = pd.read_sql(f"SELECT * FROM {username}.sbb_stops_final_region ", conn, columns=['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
rename_col(stops_df)
stop_to_stop_df = pd.read_sql(f"SELECT * FROM {username}.sbb_stop_to_stop_final_region ", conn , columns =['stop_id_a', 'stop_id_b', 'distance'])
rename_col(stop_to_stop_df)
stop_times_df = pd.read_sql(f"SELECT * FROM {username}.sbb_stop_times_final_region ", conn, columns =['trip_idddd', 'stop_id','arrival_time','departure_time'])
rename_col(stop_times_df)

route_desc_df = pd.read_csv("data/route_desc.csv")
routes_df = pd.read_csv("data/routes.csv")

We do some minor preprocessing to ensure that the time is in the right format in order to do carry out arithmatic and caluclations with the time, as will be necessary for later parts such as graph building and dijkstra. 

In [5]:
def parse_time(time_str: str = '') -> timedelta:
    """Convert time strings to timedelta"""
    hours, minutes, seconds = map(int, time_str.split(':'))
    return timedelta(hours=hours % 24, minutes=minutes, seconds=seconds)

In [4]:
stop_times_df['arrival_time'] = stop_times_df['arrival_time'].apply(parse_time)
stop_times_df['departure_time'] = stop_times_df['departure_time'].apply(parse_time)

NameError: name 'stop_times_df' is not defined

## Delay calculation
In this section of the notebook, we will be handling all of the actual data and the delay prediction .
### Spark session

In [ ]:
{ "conf": {
        "mapreduce.input.fileinputformat.input.dir.recursive": true,
        "spark.sql.extensions": "com.hortonworks.spark.sql.rule.Extensions",
        "spark.kryo.registrator": "com.qubole.spark.hiveacid.util.HiveAcidKyroRegistrator",
        "spark.sql.hive.hiveserver2.jdbc.url": "jdbc:hive2://iccluster065.iccluster.epfl.ch:2181,iccluster080.iccluster.epfl.ch:2181,iccluster066.iccluster.epfl.ch:2181/;serviceDiscoveryMode=zooKeeper;zooKeeperNamespace=hiveserver2",
        "spark.datasource.hive.warehouse.read.mode": "JDBC_CLUSTER",
        "spark.driver.extraClassPath": "/opt/cloudera/parcels/SPARK3/lib/hwc_for_spark3/hive-warehouse-connector-spark3-assembly-1.0.0.3.3.7190.2-1.jar",
        "spark.executor.extraClassPath": "/opt/cloudera/parcels/SPARK3/lib/hwc_for_spark3/hive-warehouse-connector-spark3-assembly-1.0.0.3.3.7190.2-1.jar"
    }
}
print(f"remote USER={os.getenv('USER',None)}")
print(f"local USER={os.getenv('USER',None)}")
username=os.getenv('USER', 'anonymous')
hadoop_fs=os.getenv('HADOOP_DEFAULT_FS', 'hdfs://iccluster067.iccluster.epfl.ch:8020')
print(f"local username={username}\nhadoop_fs={hadoop_fs}")


In [ ]:
username=spark.conf.get('spark.executorEnv.USERNAME', 'anonymous')
hadoop_fs=spark.conf.get('spark.executorEnv.HADOOP_DEFAULT_FS','hdfs://iccluster067.iccluster.epfl.ch:8020')
print(f"remote username={username}\nhadoop_fs={hadoop_fs}")

parquet_path = "/data/sbb/parquet/timetables/stop_times"
stop_times_df = spark.read.parquet(parquet_path)
stop_times_df.printSchema()

In [ ]:
stops_lausanne.printSchema()
stop_times_df = stop_times_df.join(stops_lausanne, "stop_id", how='inner')
stop_times_df.printSchema()

In [ ]:
stop_times_df.show(n=1, vertical=True)

indexer = StringIndexer(inputCol="stop_id", outputCol="indexed")
model = indexer.fit(stop_times_df)

stop_trip_indexed  = model.transform(stop_times_df)
stop_trip_indexed = stop_trip_indexed.withColumn("indexed", col("indexed").cast("integer")) # Cast indices to integers

# To get the stop_id back from indices'
inverter = IndexToString(inputCol="indexed", outputCol="stop_id_2", labels=model.labels) 

stop_trip_indexed.printSchema()

In [ ]:
# Self join on the df to get all the pairs of stops that share the same trip_id
joined_df = stop_trip_indexed.alias("df1").join(stop_trip_indexed.alias("df2"), "trip_id")

frequency = False # we decide that clustering using frequency as weight is good enough
if frequency: # use frequencies in trips as a weight
    # group by unique pairs of indices (pairs of stop_ids) and count how many trips go between two stops
    frequency_weight_df = joined_df.groupBy("df1.indexed", "df2.indexed").agg(count(lit(1)).alias("weight"))
    
    
    
    network_df = frequency_weight_df.filter(
        (col("df1.indexed") != col("df2.indexed"))
    ).select(
        col("df1.indexed").alias("src"),
        col("df2.indexed").alias("dst"),
        col("weight")
    )
    
    
    
    # Now we have a dataframe with src, dst, and weights but weights is just the number of trips, 
    # we will normalize the weights with spark MLlib StandardScaler
    
    assembler = VectorAssembler(
        inputCols=["weight"],
        outputCol="weight_vector"
    )
    
    # we change the dataframe to put weights as a vector, as that is the input needed for the scaler
    weight_df = assembler.transform(network_df)
    
    
    scaler = StandardScaler(
        inputCol="weight_vector",
        outputCol="scaled_weight",
        withMean=False,  # we set withMean=False so it isn't centered around 0, avoiding negative values for clustering
        withStd=True     # 
    )
    
    # Fit the scaler
    scaler_model = scaler.fit(weight_df)
    
    # Transform the df to add the scaled_weight column
    network_df = scaler_model.transform(weight_df)
    
    def extract_first_element(vector):
        return float(vector[0])
    
    # Register the UDF
    extract_first_element_udf = udf(extract_first_element, FloatType())
    
    # Apply the UDF to create a new column containing the first element of the vector as a float
    network_df = network_df.withColumn("scaled_weight_float", extract_first_element_udf(col("scaled_weight")))
    
    
    
    # TODO: create network_df (hint: cast stop indices to integers)
    network_df = network_df.select(
        col('src'),
        col('dst'),
        col('scaled_weight_float').alias('weight'))
    
    
else: # else use differences in stop_sequences 
    # Here we create another df to test another metric for the clustering 
    sequence_weight_df = joined_df \
        .withColumn("sequence_weight", 1/(F.abs(joined_df["df1.stop_sequence"] - joined_df["df2.stop_sequence"]) ))
    

    network_df = sequence_weight_df.filter(
        (col("df1.indexed") != col("df2.indexed"))
    ).select(
        col("df1.indexed").alias("src"),
        col("df2.indexed").alias("dst"),
        col("sequence_weight").alias("weight")
    ).dropDuplicates(['src', 'dst'])

In [ ]:
if frequency:
    best_k = 7 # For data only in Lausanne area, with frequency weight
else: # weights based on trip sequences 
    # The code to find the best_k is the same and written below, but for time reasons and cluster usage
    # we won't run it for the different type of weight and assume a best_k of 10 for lausanne region
    best_k = 10

# Set this to true to find optimal clustering. It takes a long time
FIND_BEST_K = False
if FIND_BEST_K: 
    best_k =0
    best_modularity = -1
    for k in range(3,13): # 
        print(f"clustering with k = {k} ...")
        pic = PowerIterationClustering(k=k,srcCol='src',dstCol='dst', weightCol="weight")
        pic.setMaxIter(20)
        #model_pic = pic.fit(network_df)
        # assign clusters in the network dataframe
        assignments = pic.assignClusters(network_df)
        
        
        
        
        m = network_df.count()
        
        # Compute sum of weights attached to each node (di and dj)
        di_df = network_df.groupBy("src").agg(F.sum("weight").alias("di"))
        dj_df = network_df.groupBy("dst").agg(F.sum("weight").alias("dj"))
        
        
        
        # Compute number of edges between each pair of nodes (eij)
        eij_df = network_df.groupBy("src", "dst").count().withColumnRenamed("count", "eij")
        
        # Put everything in one dataframe
        joined_df = network_df.join(assignments, network_df['src'] == assignments['id'], "left") \
            .withColumnRenamed("cluster", "cluster_i") \
            .drop(assignments['id']) \
            .join(assignments, network_df.dst == assignments['id'], "left") \
            .withColumnRenamed("cluster", "cluster_j") \
            .drop(assignments['id'])
        
        joined_df = joined_df.join(eij_df, (joined_df['src'] == eij_df['src']) & (joined_df['dst'] == eij_df['dst']), "left").drop(eij_df['src']).drop(eij_df['dst'])
        joined_df = joined_df.join(di_df, joined_df['src'] == di_df['src'], "left").drop(di_df['src'])
        joined_df = joined_df.join(dj_df, joined_df['dst'] == dj_df['dst'], "left").drop(dj_df['dst'])
        
        #Compute modularity 
        modularity_df = joined_df \
            .withColumn("same_cluster", F.when(F.col("cluster_i") == F.col("cluster_j"), 1).otherwise(0)) \
            .groupBy().agg(
                F.sum(F.col("same_cluster") * F.col("eij") / m - F.col("di") * F.col("dj") / (2 * m * m)).alias("modularity")
            )
        
        # Show result
        #modularity_df.show(1)
        
        modularity = modularity_df.collect()[0]['modularity']
        print(f"Modularity : {modularity}")
        if modularity* 1.01 > best_modularity :
            best_modularity = modularity
            best_k = k
        else: # when the modularity does not improve, get out of the loop
            print(best_k, best_modularity)
            break

In [ ]:
pic = PowerIterationClustering(k=best_k,srcCol='src',dstCol='dst', weightCol="weight")
pic.setMaxIter(20)
# assign clusters in the network dataframe
assignments = pic.assignClusters(network_df)

In [ ]:
assignments.printSchema()
inverter = IndexToString(inputCol="id", outputCol="stop_id", labels=model.labels) 
assignments = inverter.transform(assignments)

In [ ]:
stops_df = spark.read.parquet("/data/sbb/parquet/timetables/stops/year=2024/month=1/day=10")
stop_cluster_loc_df = assignments.join(stops_df, assignments['stop_id'] == stops_df['stop_id'], how='inner')

stop_cluster_loc_df.printSchema()

In [ ]:
fig = px.scatter_mapbox(stop_cluster_loc_df, lat='stop_lat', lon='stop_lon', color='cluster', title='Map of Clusters',
                        labels={'stop_lon': 'Longitude', 'stop_lat': 'Latitude'}, zoom=10)
fig.update_layout(mapbox_style="open-street-map")


fig.show()


### Model creation

#### Data sanitizing

In [ ]:
stop_cluster_loc_df.show(2)


stations_df = spark.read.csv('/data/wunderground/csv/stations', header=True)
trips_df = spark.read.orc('/data/sbb/orc/istdaten')

# Renaming columns with English translations
trips_df = (trips_df
            .withColumnRenamed("betriebstag", "operation_day")
            .withColumnRenamed("fahrt_bezeichner", "trip_identifier")
            .withColumnRenamed("betreiber_id", "operator_id")
            .withColumnRenamed("betreiber_abk", "operator_abbreviation")
            .withColumnRenamed("betreiber_name", "operator_name")
            .withColumnRenamed("produkt_id", "product_id")
            .withColumnRenamed("linien_id", "line_id")
            .withColumnRenamed("linien_text", "line_text")
            .withColumnRenamed("umlauf_id", "rotation_id")
            .withColumnRenamed("verkehrsmittel_text", "transportation_text")
            .withColumnRenamed("zusatzfahrt_tf", "additional_trip_tf")
            .withColumnRenamed("faellt_aus_tf", "is_cancelled_tf")
            .withColumnRenamed("bpuic", "station_code")
            .withColumnRenamed("haltestellen_name", "stop_name")
            .withColumnRenamed("ankunftszeit", "arrival_time")
            .withColumnRenamed("an_prognose", "arrival_forecast")
            .withColumnRenamed("an_prognose_status", "arrival_forecast_status")
            .withColumnRenamed("abfahrtszeit", "departure_time")
            .withColumnRenamed("ab_prognose", "departure_forecast")
            .withColumnRenamed("ab_prognose_status", "departure_forecast_status")
            .withColumnRenamed("durchfahrt_tf", "transit_tf")
            .withColumnRenamed("year", "year")
            .withColumnRenamed("month", "month")
           )

# Clean up null values in trips_df
trips_df = trips_df.na.drop()

# Ignore cancelled trips
filtered_trips_df = trips_df.filter(trips_df["is_cancelled_tf"] == "false")

# Selecting only relevant columns based on the delay detection requirement
columns_to_keep = ["product_id", 
    "arrival_time", "arrival_forecast",
    "departure_time", "departure_forecast"
]

# Filtering the DataFrame to keep only the necessary columns
filtered_trips_df = filtered_trips_df.select(columns_to_keep)

# Constructing a dynamic filter to exclude empty strings in these columns
non_empty_filters = [col(column_name) != "" for column_name in columns_to_keep]

# Combine all conditions into one
combined_filter = reduce(lambda a, b: a & b, non_empty_filters, lit(True))

# Apply the combined filter to the DataFrame
filtered_trips_df = filtered_trips_df.filter(combined_filter)

casted_trips_df = filtered_trips_df

time_format = "dd.MM.yyyy HH:mm"
forecast_format = "dd.MM.yyyy HH:mm:ss"

# Convert time columns to timestamp type using the specified format
casted_trips_df = casted_trips_df.withColumn("arrival_time", to_timestamp("arrival_time", time_format))
casted_trips_df = casted_trips_df.withColumn("arrival_forecast", to_timestamp("arrival_forecast", forecast_format))
casted_trips_df = casted_trips_df.withColumn("departure_time", to_timestamp("departure_time", time_format))
casted_trips_df = casted_trips_df.withColumn("departure_forecast", to_timestamp("departure_forecast", forecast_format))

# Show the updated schema to verify changes
casted_trips_df.printSchema()

# Calculate arrival and departure delays in minutes
casted_trips_df = casted_trips_df.withColumn(
    "arrival_delay_minutes",
    (unix_timestamp("arrival_forecast") - unix_timestamp("arrival_time")) / 60
)

#### Feature Vector Construction using Spark MLlib


In [ ]:
# Sample approximately 10% of the data without replacement (due to resources)
df = casted_trips_df.sample(False, 0.1, seed=42)

df = df.withColumn("label", when(df.arrival_delay_minutes <= 5, 0)
                   .otherwise(1))

# Extracting features from timestamps
df = df.withColumn("arrival_hour", hour(df.arrival_time))
df = df.withColumn("departure_hour", hour(df.departure_time))
df = df.withColumn("day_of_week", dayofweek(df.arrival_time))

# Define the feature columns and VectorAssembler
feature_columns = ['arrival_hour', 'departure_hour', 'day_of_week']
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Apply the VectorAssembler to transform the data
vectorized_df = assembler.transform(df)

# Define the StandardScaler
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

# Create a Pipeline that includes both VectorAssembler and StandardScaler
pipeline = Pipeline(stages=[assembler, scaler])

# Fit the Pipeline to your data
pipelineModel = pipeline.fit(df)

# Save the Pipeline model to a directory
pipeline_path = "pipeline/delayed_trips"
pipelineModel.write().overwrite().save(pipeline_path)


# Apply the Pipeline model to transform the data
transformed_df = pipelineModel.transform(df)

# Split the data into training and validation sets
train_data, validation_data = transformed_df.randomSplit([0.8, 0.2], seed=42)


# Train the Logistic Regression model
lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="label")
lr_model = lr.fit(train_data)

# Save the model
model_path = "model/logisitic_regression"
lr_model.write().overwrite().save(model_path)

#### Model evaluation

In [ ]:


# Creating the evaluator with default metric (accuracy)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# Evaluating accuracy
accuracy = evaluator.evaluate(lr_model.transform(validation_data))
print(f"Accuracy: {accuracy:.2f}")

# Evaluating precision
evaluator.setMetricName("weightedPrecision")
precision = evaluator.evaluate(lr_model.transform(validation_data))
print(f"Precision: {precision:.2f}")

# Evaluating recall
evaluator.setMetricName("weightedRecall")
recall = evaluator.evaluate(lr_model.transform(validation_data))
print(f"Recall: {recall:.2f}")

# Evaluating F1-score
evaluator.setMetricName("f1")
f1 = evaluator.evaluate(lr_model.transform(validation_data))
print(f"F1 Score: {f1:.2f}")

In [ ]:
def make_prediction(spark, sample, pipeline_path="pipeline/delayed_trips", model_path="model/logisitic_regression"):
    """
    Predict the probability of delay using a saved feature transformation pipeline and logistic regression model.
    
    Args:
        spark (SparkSession): The active Spark session.
        sample (tuple): Sample data as a tuple with fields (product_id, arrival_time, departure_time, day_of_week).
        pipeline_path (str): The path to the saved pipeline model for feature processing.
        model_path (str): The path to the saved logistic regression model for prediction.
        
    Returns:
        float: The probability of delay for the sample.
    """
    # Load the pipeline and model
    pipeline_model = PipelineModel.load(pipeline_path)
    logistic_model = LogisticRegressionModel.load(model_path)
    
    # Define schema based on the expected input for the model
    schema = StructType([
        StructField("product_id", StringType(), True),
        StructField("arrival_time", StringType(), True),
        StructField("departure_time", StringType(), True),
    ])
    
    # Create a DataFrame from the sample
    sample_df = spark.createDataFrame([sample], schema=schema)

    # Convert string timestamps to TimestampType
    sample_df = sample_df.withColumn("arrival_time", to_timestamp("arrival_time"))
    sample_df = sample_df.withColumn("departure_time", to_timestamp("departure_time"))

    # Calculate additional necessary columns if needed
    sample_df = sample_df.withColumn("arrival_hour", hour("arrival_time"))
    sample_df = sample_df.withColumn("departure_hour", hour("departure_time"))
    sample_df = sample_df.withColumn("day_of_week", dayofweek(df.arrival_time))

    # Transform the sample using the pipeline
    transformed_sample = pipeline_model.transform(sample_df)

    # Make a prediction
    prediction = logistic_model.transform(transformed_sample)
    
    # Extract the probability of delay
    probability_of_delay = prediction.select("probability").head()[0][1]
    
    return probability_of_delay

### Brief overview of algorithm
Our approach is split into 2 parts, creating a graph, and then running dijkstra's on the graph.




## Building the graph
Idea of graph - each node represents a stop and edges between nodes represent a direct trip between stops. The graph is directed.
For each stop, determine and sort the next possible departure times so that it is easy to look-up connection times.
Iterate through each trip, and calculate the travel time, and wait time store, these in the edges.
- Travel time: The difference between the departure time at the first stop to the arrival time of the next stop. 
- Wait Time: The time until the next trip departs from the stop, taking into account edge cases where the next trip may be on the next day. 

In [6]:
def calculate_walking_time(distance_meters: float = 0.0) -> timedelta:
    """
    Calculate the walking timedelta of a given distance.
    We assumed that a normal person can walk 50 meters in 1 minute.
    Otherwise, we assumed that the walking timedelta is 2 minutes
    """
    return max(timedelta(minutes=distance_meters / 50), timedelta(minutes=2))

In [11]:
def build_graph(stop_times_df: DataFrame = None,
                stops_df: DataFrame = None,
                stop_to_stop_df: DataFrame = None
                ) -> Dict[str, List[Tuple[str, timedelta, timedelta, timedelta, str]]]:
    """
    Builds a graph representing connections between public transport stops and walking paths.

    Args:
        stop_times_df (DataFrame): Contains information about the stop times for each trip.
        stops_df (DataFrame): Contains information about each stop.
        stop_to_stop_df (DataFrame): Contains information about the walking distances between stops.

    Returns:
        Dict[str, List[Tuple[str, timedelta, timedelta, timedelta, str]]]: A dictionary where each key is a stop_id_A, 
        and the value is a List of Tuples. Each tuple represents a connection to another stop_id_B, with information on 
        travel_time, wait_time, departure_time, and trip_id.
    """
    # Initialize the graph with an empty list for each stop
    graph = {stop: [] for stop in stops_df['stop_id']}
    next_departures = {}

    # Process stop times for public transport connections
    for stop_id, group in stop_times_df.groupby('stop_id'):
        sorted_group = group.sort_values('departure_time')
        # Record the next departure times for each stop and trip combination
        next_departures[stop_id] = {row['trip_id']: row['departure_time'] for _, row in sorted_group.iterrows()}

    # Add connections between stops based on trip_id
    for _, group in stop_times_df.groupby('trip_id'):
        # Sort each group by departure_time
        sorted_group = group.sort_values('departure_time')
        # Iterate through each sorted group to calculate travel time and wait time
        for i in range(len(sorted_group) - 1):
            row_current = sorted_group.iloc[i]
            row_next = sorted_group.iloc[i + 1]
            stop_a = row_current['stop_id']
            stop_b = row_next['stop_id']
            trip_id = row_current['trip_id']  
            arrival_time = row_current['arrival_time']
            departure_time = row_next['departure_time']
            travel_time = departure_time - arrival_time
            wait_time = (next_departures.get(stop_b, {}).get(trip_id, departure_time) - arrival_time)
            
             # Adjust for wait_time by adding a day
            if wait_time < timedelta(0):
                wait_time += timedelta(days=1)
            
            # Add the connection between the stop_id_A and the stop_id_B to the graph
            graph[stop_a].append((stop_b, travel_time, wait_time, departure_time, trip_id))

    # Add walking connections without a trip ID (or use None)
    for _, row in stop_to_stop_df.iterrows():
        stop_a = row['stop_id_a']
        stop_b = row['stop_id_b']
        distance = row['distance']
        # Calculate walking time from distance
        walking_time = calculate_walking_time(distance)
        # Add the walking connection to the graph with default wait_time, departure_time, trip_id values
        graph[stop_a].append((stop_b, walking_time, timedelta(0), None, None))
        graph[stop_b].append((stop_a, walking_time, timedelta(0), None, None))
        
    # Return the completed graph
    return graph

In [ ]:
# WARNING: Building the graph below can take up to 1 minute
# Create the graph instance
graph = build_graph(stop_times_df, stops_df, stop_to_stop_df)

In [13]:
def get_probability_of_train_being_on_time(start_stop_id: str = '',
                                           end_stop_id: str = '',
                                           schedule_departure_time: timedelta = None,
                                           schedule_arrival_time: timedelta = None,
                                           transport_type: str = '',
                                           date: str = '',
                                           wait_time: float = 0.0) -> float:
    """In Switzerland, (assumption) trains are on time 100% of the time"""
    return 100.0

### Dijkstra:
For each node you visit, consider all outgoing edges and calculate the new arrival time at the next node by summing the current node's arrival time, wait time and travel time.
Update the shortest travel time to each node if a faster route is found. 

In [14]:
def dijkstra_time_aware(graph: Dict[str, List[Tuple[str, timedelta, timedelta, timedelta, str]]] = {},
                        start_id: str = '', end_id: str = '', start_time: str = '',
                        threshold: float = 0.0
                        ) -> List[Tuple[str, float, str]]:
    """
    Computes the shortest path in a time-aware manner using Dijkstra's algorithm.

    Args:
        graph (Dict[str, List[Tuple[str, timedelta, timedelta, timedelta, str]]]): The graph representing the network of stops.
        start_id (str): The starting stop_id_A.
        end_id (str): The ending stop_id_B.
        start_time (str): The start time in a string format.
        threshold (float): The minimum probability threshold for considering a connection.

    Returns:
        List[Tuple[str, float, str]]: The shortest path from start to end, including stop_id, travel time in minutes, and trip type.
    """
    pq = [(start_time, start_id, [], None)] 
    visited = {}

    while pq:
        # Pop the stop with the earliest time
        current_time, current_stop, path, last_trip_id = heapq.heappop(pq)
        
        # If the current stop is the end stop, return the path
        if current_stop == end_id:
            return [(stop, (time - start_time).total_seconds() / 60, trip_id if trip_id else "walking")
                    for stop, time, trip_id in path]
        
        # Skip this stop if it has already been visited with an earlier or equal time
        if current_stop in visited and visited[current_stop] <= current_time:
            continue
        visited[current_stop] = current_time
        
        # Process all neighbors of the current stop
        for next_stop, travel_time, _, departure_time, next_trip_id in graph[current_stop]:
            if departure_time is None:
                # If there is no departure_time, then this is a walking connection
                departure_time_adjusted = current_time
                current_trip_id = "walking"
                probability = 97.5 # Certainty of complete the walking path on time
            else:
                # Skip if the current time is after the departure time
                if current_time > departure_time:
                    continue
                departure_time_adjusted = max(departure_time, current_time)
                current_trip_id = next_trip_id
                
                # Get the probability of the train being on time
                probability = get_probability_of_train_being_on_time(current_stop, next_stop, departure_time, 
                                                                     departure_time + travel_time,
                                                                     "Underground", "2024-01-01", 2) 
            
            # Skip this connection if the probability is below the threshold
            if probability < threshold:
                continue  
            
            # Adjust for transfer time if changing trips
            if next_trip_id is not None and last_trip_id != next_trip_id:
                departure_time_adjusted = max(departure_time_adjusted, current_time + timedelta(minutes=2))
            
            # Calculate the new time after taking this connection
            new_time = departure_time_adjusted + travel_time
            new_path = path + [(next_stop, new_time, current_trip_id)]
            
            # Add to the priority queue if not visited or found a shorter path
            if next_stop not in visited or new_time < visited[next_stop]:
                heapq.heappush(pq, (new_time, next_stop, new_path, next_trip_id))
    
    # Return an empty list if no path is found
    return []

In [15]:
# Initial stop and time
#start_stop_id = '8501214'
#start_time = timedelta(hours=0)
#end_stop_id = '8588158'

## Post-Processing
The output from dijkstra is not exactly in the format needed for the UI, we need to parse the output of dijkstra to be mappable and turned into the format of an itinerary. We also parse the input provided by user to the widgets so that they can be fed to dijkstra. 

In [16]:
def parse_input(input: DataFrame = None) -> Dict:
    """
    Parses the input from the GUI to a Dictionary with the desired Keys for further processing.
    
    Args:
        input: Dataframe of one entry with the initial and destination stops ids, the starting time, and the confidence of the trip 
    Returns:
        Dictionary with the desired Keys:
            'start_stop_id': (str) start_stop_id
            'start_time': (timedelta) parsed_time
            'end_stop_id': (str) end_stop_id
            'confidence': (float) confidence        
     """
    start_stop_id = stops_df.loc[stops_df['stop_name'] == input['from'], 'stop_id'].iloc[0]
    end_stop_id = stops_df.loc[stops_df['stop_name'] == input['to'], 'stop_id'].iloc[0]
    confidence = input['confidence']

    if input['time']:
        parsed_time = timedelta(hours=input['time'].hour, minutes=input['time'].minute)
    else:
        parsed_time = timedelta(hours=0) 

    parsed_input = {
        'start_stop_id': start_stop_id,
        'start_time': parsed_time,
        'end_stop_id': end_stop_id,
        'confidence': confidence
    }
    return parsed_input

In [17]:
def extract_route_id(trip_id: str = '') -> str:
    """Check if it is a standard route_id and return it or None"""
    match = re.search(r'TA\.(\d+-[A-Za-z0-9-]+-j24-\d+)', trip_id)
    if match:
        return match.group(1)
    return None

In [18]:
def parse_output(graph: Dict = {},
                 start_id: str = '', end_id: str = '', start_time: str = '',
                 routes_df: DataFrame = None, route_desc_df: DataFrame = None, stops_df: DataFrame = None,
                 threshold: float = 0.0
                 ) -> DataFrame:
    """
    Parses the output of the dijkstra_time_aware function and returns detailed journey information.

    Args:
        graph (Dict): The graph representing the network of stops.
        start_id (str): The starting stop ID.
        end_id (str): The ending stop ID.
        start_time (str): The start time in a string format.
        routes_df (DataFrame): DataFrame containing route information.
        route_desc_df (DataFrame): DataFrame containing route descriptions.
        stops_df (DataFrame): DataFrame containing stop information.
        threshold (float): The minimum probability threshold for considering a connection.

    Returns:
        DataFrame: A DataFrame containing detailed journey information.
    """
    path = dijkstra_time_aware(graph, start_id, end_id, start_time, threshold)
    journey_details = []

    print("\nJourney Details:\n")

    if path:
        # Process the first stop
        first_stop, first_time, first_trip_id = path[0]
        first_time_delta = timedelta(minutes=first_time)
        first_adjusted_time = (start_time + first_time_delta)
        
        # Get information about the first stop
        first_stop_info = stops_df[stops_df['stop_id'] == start_id]  
        if not first_stop_info.empty:
            first_stop_name = first_stop_info['stop_name'].values[0]
            first_stop_lat = first_stop_info['stop_lat'].values[0]
            first_stop_lon = first_stop_info['stop_lon'].values[0]
        else:
            first_stop_name, first_stop_lat, first_stop_lon = "Unknown stop", None, None
        
        # Determine the mode of transportation for the first segment
        if first_trip_id == "walking":
            first_transport_mode = "Walk"
            first_adjusted_time = start_time
        else:
            first_route_id = extract_route_id(first_trip_id)
            first_route_type = routes_df[routes_df['route_id'] == first_route_id]['route_type'].iloc[0] if not routes_df[routes_df['route_id'] == first_route_id].empty else None
            first_transport_mode = route_desc_df[route_desc_df['route_type'] == first_route_type]['EN'].iloc[0] if first_route_type and not route_desc_df[route_desc_df['route_type'] == first_route_type].empty else "Unknown mode"
        
        # Append details of the first stop to journey_details
        journey_details.append({
            "stop_id": start_id,
            "stop_name": first_stop_name,
            "stop_lat": first_stop_lat,
            "stop_lon": first_stop_lon,
            "time_at_stop": first_adjusted_time,
            "form_of_transportation": first_transport_mode
        })

        print(f"{first_transport_mode} from Starting Location: {first_stop_name}, Time at stop: {first_adjusted_time}")
    
    # Process subsequent stops in the path
    for index, (stop, time, trip_id) in enumerate(path[1:], start=1):
        current_time = timedelta(minutes=time)
        adjusted_time = (start_time + current_time)
        stop_info = stops_df[stops_df['stop_id'] == stop]
        
        # Get information about the current stop or fill the variables with default values
        if not stop_info.empty:
            stop_name = stop_info['stop_name'].values[0]
            stop_lat = stop_info['stop_lat'].values[0]
            stop_lon = stop_info['stop_lon'].values[0]
        else:
            stop_name, stop_lat, stop_lon = "Unknown stop", None, None
        
        # Determine the mode of transportation for the current segment
        if trip_id == "walking":
            transport_mode = "Walk"
        else:
            route_id = extract_route_id(trip_id)
            route_type = routes_df[routes_df['route_id'] == route_id]['route_type'].iloc[0] if not routes_df[routes_df['route_id'] == route_id].empty else None
            transport_mode = route_desc_df[route_desc_df['route_type'] == route_type]['EN'].iloc[0] if route_type and not route_desc_df[route_desc_df['route_type'] == route_type].empty else "Unknown mode"

        print(f"{transport_mode} to {stop_name}, Time at stop: {adjusted_time}")
        
        # Append details of the current stop to journey_details
        journey_details.append({
            "stop_id": stop,
            "stop_name": stop_name,
            "stop_lat": stop_lat,
            "stop_lon": stop_lon,
            "time_at_stop": adjusted_time,
            "form_of_transportation": transport_mode
        })
    # Format the journey details as a DataFrame and return it to be visualised on GUI
    journey_details_df = pd.DataFrame(journey_details)
    return journey_details_df

In [19]:
def plot_route(df):
    # Create a color map for unique forms of transportation
    unique_transportation = df['form_of_transportation'].unique()
    color_map = {transport: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)] 
                 for i, transport in enumerate(unique_transportation)}

    # Create the figure
    fig = go.Figure()

    # Track which transport forms have been added to the legend
    transports_added = {}

    # Iterate over the DataFrame and draw lines and markers between each row
    for i in range(len(df)):
        transport_type = df.iloc[i]['form_of_transportation']
        show_legend = not transports_added.get(transport_type, False)
        
        if i < len(df) - 1:
            # Add lines connecting the stops
            fig.add_trace(go.Scattermapbox(
                mode="lines",
                lon=[df.iloc[i]['stop_lon'], df.iloc[i + 1]['stop_lon']],
                lat=[df.iloc[i]['stop_lat'], df.iloc[i + 1]['stop_lat']],
                line={'width': 2, 'color': color_map[transport_type]},
                name=transport_type,
                showlegend=show_legend
            ))
        
        # Add markers for each stop
        fig.add_trace(go.Scattermapbox(
            mode="markers+text",
            lon=[df.iloc[i]['stop_lon']],
            lat=[df.iloc[i]['stop_lat']],
            text=[df.iloc[i]['time_at_stop'].total_seconds()//3600],  # Display time in hours
            textposition="bottom right",
            marker={'size': 10, 'color': color_map[transport_type]},
            name=transport_type,
            showlegend=False
        ))

        # Mark this transport form as added to the legend
        transports_added[transport_type] = True

    # Update the layout
    fig.update_layout(
        margin={'l': 0, 't': 0, 'b': 0, 'r': 0},
        mapbox={
            'style': "open-street-map",
            'center': {'lon': df['stop_lon'].mean(), 'lat': df['stop_lat'].mean()},
            'zoom': 10  # Adjust zoom level as needed
        }
    )

    fig.show()

## UI and Applet

skididi

In [ ]:
# Initial stop and time

# You need at a minimum to allow us to enter a starting point an end point, a maximum expected time of arrival, and a minimum confidence of arriving before the maximum expected time.
# Optionally, if you support any day of the week (Monday to Sunday), you should allow us to pick one of the supported day. Otherwise, be specific of which day of the week the journey is valid so that we only compare with SBB's journey for that day.
# Do not specify a week of year (just use a recent timetable, e.g. 2024.5.16 which is available from the com490final db.
# It must be interactive, so that we can try different routes / time of day.

# Inputs: 
# - starting point
# - endpoint
# - (expected time of arrival)
# - a minimum confidence of arriving before the maximum expected time
# - Date 

# Outputs (after algorithm):
# The trajectory (Flon -> Montelly with metro -> walk to Malley -> ...)
# Can be done also via map and with ID's and so on..




global stops_df

#print(stops_df)
stop_list = sorted(stops_df['stop_name'].tolist())

# Create the first dropdown widget
from_dropdown = widgets.Dropdown(
    options=stop_list,
    value=stop_list[0],
    description='From:',
    disabled=False
)

# Create the second dropdown widget
to_dropdown = widgets.Dropdown(
    options=stop_list,
    value=stop_list[100],
    description='To:',
    disabled=False
)

## We might want to include day as well!
# Create a time selector widget
time_picker = widgets.TimePicker(
    description='Departure:',
    disabled=False
)

# Create a float slider widget
float_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=100.0,
    step=0.1,
    description='Confidence:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Create a button widget
button = widgets.Button(
    description='Find Route',
    disabled=False,
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)

# Create an output area
output = widgets.Output()



# Define a callback function for the button click event
def on_button_clicked(b):
    global start_stop_id, start_time, end_stop_id 

    with output:
        output.clear_output()
        input = {
            'from': from_dropdown.value,
            'to': to_dropdown.value,
            'time': time_picker.value,
            'confidence': float_slider.value
        }
        parsed_input = parse_input(input)

        start_stop_id = parsed_input['start_stop_id']
        start_time = parsed_input['start_time']
        end_stop_id = parsed_input['end_stop_id']

        df = parse_output(graph, start_stop_id, end_stop_id, start_time, routes_df, route_desc_df,stops_df, threshold=parsed_input['confidence'])
        plot_route(df)
        

# Attach the callback function to the button
button.on_click(on_button_clicked)

# Display the widgets
display(from_dropdown, to_dropdown, time_picker, float_slider, button, output)

# You need at a minimum to show the details of the public transport journey (including walking if any). 
# It must be presented in a way that can be easily compared with SBB, e.g. list of human readable stop names
# and arrival departures, or list of public transport line ID or numbers that can be easily compared with 
# SBB, or map information showing the stops on a map and different transports (e.g. using different colors).
